# Часть 1


### Шаг 1. Подготовка данных: Матрица взаимодействий и признаки

In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix
from typing import Tuple

# Специфичные колонки твоего датасета
COL_USER = 'Customer_Name'
COL_ITEM = 'Product'
COL_QTY = 'Total_Items'
COL_DATE = 'Date'
COL_COST = 'Total_Cost'
COL_DISCOUNT = 'Discount_Applied'

def prepare_retail_data(file_path: str) -> Tuple[csr_matrix, pd.DataFrame, pd.DataFrame]:
    """
    Загружает данные и готовит компоненты для рекомендательной системы.
    """
    # 1. Загрузка
    df = pd.read_csv(file_path)
    df[COL_DATE] = pd.to_datetime(df[COL_DATE])
    
    # --- 2. Признаки покупателей (User Features) ---
    print("Генерируем признаки пользователей...")
    user_features = df.groupby(COL_USER).agg(
        # Косвенный признак 1: Средний чек
        avg_spend=(COL_COST, 'mean'),
        # Косвенный признак 2: Склонность к скидкам (доля покупок со скидкой)
        discount_affinity=(COL_DISCOUNT, lambda x: x.map({True: 1, False: 0}).mean()),
        # Базовый признак: Общее кол-во транзакций
        total_transactions=('Transaction_ID', 'nunique')
    ).reset_index()

    # --- 3. Признаки товаров (Product Features) ---
    print("Генерируем признаки товаров...")
    item_features = df.groupby(COL_ITEM).agg(
        # Признак 1: Общая выручка от товара
        total_revenue=(COL_COST, 'sum'),
        # Признак 2: Популярность (сколько раз попадал в чеки)
        popularity=(COL_QTY, 'sum'),
        # Косвенный признак: "Рейтинг" на основе частоты упоминаний в разных категориях
        unique_categories=('Customer_Category', 'nunique')
    ).reset_index()

    # --- 4. Матрица взаимодействий ---
    print("Строим матрицу взаимодействий...")
    # Группируем, чтобы получить суммарное кол-во купленного товара каждым юзером
    interactions_df = df.groupby([COL_USER, COL_ITEM])[COL_QTY].sum().reset_index()

    # Создаем категориальные индексы (Mapping)
    df[COL_USER] = df[COL_USER].astype('category')
    df[COL_ITEM] = df[COL_ITEM].astype('category')

    # Важно: используем те же коды, что и в основном DF
    row_indices = df[COL_USER].cat.codes
    col_indices = df[COL_ITEM].cat.codes
    values = df[COL_QTY]

    # Строим разреженную матрицу CSR
    # [Users, Items]
    interaction_matrix = csr_matrix(
        (values, (row_indices, col_indices)),
        shape=(len(df[COL_USER].cat.categories), len(df[COL_ITEM].cat.categories))
    )

    return interaction_matrix, user_features, item_features

# Запуск
interaction_matrix, user_feats, item_feats = prepare_retail_data('data/retail.csv')

print(f"\nГотово!")
print(f"Матрица: {interaction_matrix.shape[0]} пользователей на {interaction_matrix.shape[1]} товаров.")
print(f"Признаков пользователей: {user_feats.shape[1] - 1}")
print(f"Признаков товаров: {item_feats.shape[1] - 1}")

Генерируем признаки пользователей...
Генерируем признаки товаров...
Строим матрицу взаимодействий...

Готово!
Матрица: 329738 пользователей на 571947 товаров.
Признаков пользователей: 3
Признаков товаров: 3


In [8]:
def verify_integrity(df: pd.DataFrame, interaction_matrix: csr_matrix):
    unique_pairs_in_df = df.groupby([COL_USER, COL_ITEM]).size().shape[0]
    
    unique_pairs_in_matrix = interaction_matrix.nnz
    
    total_qty_df = df[COL_QTY].sum()
    total_qty_matrix = interaction_matrix.sum()
    
    print("--- Проверка целостности данных ---")
    print(f"Уникальных пар (User-Item) в DF: {unique_pairs_in_df}")
    print(f"Уникальных пар (User-Item) в Матрице: {unique_pairs_in_matrix}")
    
    if unique_pairs_in_df == unique_pairs_in_matrix:
        print("✅ Уникальные пары совпадают. Данные не потеряны.")
    else:
        print("⚠️ ВНИМАНИЕ: Расхождение в парах!")

    print(f"Общее кол-во товаров в DF: {total_qty_df}")
    print(f"Общее кол-во товаров в Матрице: {total_qty_matrix}")
    
    if np.isclose(total_qty_df, total_qty_matrix):
        print("✅ Общее количество товаров совпадает. Агрегация верна.")
    else:
        print("⚠️ ВНИМАНИЕ: Расхождение в общем количестве!")
raw_data = pd.read_csv('data/retail.csv')
# Запусти проверку
verify_integrity(raw_data, interaction_matrix)

--- Проверка целостности данных ---
Уникальных пар (User-Item) в DF: 996402
Уникальных пар (User-Item) в Матрице: 996402
✅ Уникальные пары совпадают. Данные не потеряны.
Общее кол-во товаров в DF: 5495941
Общее кол-во товаров в Матрице: 5495941
✅ Общее количество товаров совпадает. Агрегация верна.


In [19]:
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
import pandas as pd

# ---------------------------------------------------------
# Шаг 2 (Исправленный): ОБУЧЕНИЕ БЕЗ ТРАНСПОНИРОВАНИЯ
# ---------------------------------------------------------
print("1. обучаем ALS (современный синтаксис Users x Items)...")
als_model = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=30, random_state=42)
# Передаем матрицу КАК ЕСТЬ!
als_model.fit(interaction_matrix)

print("2. обучаем BPR (современный синтаксис Users x Items)...")
bpr_model = BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=100, random_state=42)
# Передаем матрицу КАК ЕСТЬ!
bpr_model.fit(interaction_matrix)

# ---------------------------------------------------------
# Шаг 3 (Исправленный): ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ
# ---------------------------------------------------------
def get_recommendations_clean(model, user_label: str, full_matrix, user_map: dict, item_map: dict, n: int = 5):
    inv_user_map = {v: k for k, v in user_map.items()}
    user_idx = inv_user_map.get(user_label)
    
    if user_idx is None:
        return None

    # В современной версии: передаем user_idx и ровно 1 строку из матрицы
    ids, scores = model.recommend(
        userid=user_idx, 
        user_items=full_matrix[user_idx], 
        N=n, 
        filter_already_liked_items=True
    )
    
    decoded_items = [item_map[i] for i in ids]
    
    return pd.DataFrame({
        'Product': decoded_items,
        'Score': scores
    })

print("\n3. Генерация рекомендаций...\n")

# Восстанавливаем словари
user_categories = raw_data[COL_USER].astype('category').cat.categories
item_categories = raw_data[COL_ITEM].astype('category').cat.categories
user_map = dict(enumerate(user_categories))
item_map = dict(enumerate(item_categories))

# Берем 5 тестовых пользователей
test_users = user_categories[:5].tolist()

for user in test_users:
    print(f"👤 Пользователь: {user}")
    
    res_als = get_recommendations_clean(als_model, user, interaction_matrix, user_map, item_map, n=5)
    res_bpr = get_recommendations_clean(bpr_model, user, interaction_matrix, user_map, item_map, n=5)
    
    if res_als is not None and res_bpr is not None:
        res_als.columns = ['ALS_Product', 'ALS_Score']
        res_bpr.columns =['BPR_Product', 'BPR_Score']
        display(pd.concat([res_als, res_bpr], axis=1))
    print("-" * 60)

1. обучаем ALS (современный синтаксис Users x Items)...


  0%|          | 0/30 [00:00<?, ?it/s]

2. обучаем BPR (современный синтаксис Users x Items)...


  0%|          | 0/100 [00:00<?, ?it/s]


3. Генерация рекомендаций...

👤 Пользователь: Aaron Acevedo


...

------------------------------------------------------------
👤 Пользователь: Aaron Adkins


,ALS_Product,ALS_Score,BPR_Product,BPR_Score
0,['Onions'],0.104262,"['Shampoo', 'Tuna']",0.443742
1,['Apple'],0.089176,"['Dishware', 'Tissues']",0.405673
2,['Rice'],0.076319,"['Canned Soup', 'Salmon']",0.389529
3,['Peanut Butter'],0.075023,"['Toothpaste', 'Toothpaste']",0.379557
4,['Mayonnaise'],0.072744,"['Pasta', 'Olive Oil']",0.363850


------------------------------------------------------------
👤 Пользователь: Aaron Aguilar


,ALS_Product,ALS_Score,BPR_Product,BPR_Score
0,['Ketchup'],0.143572,"['Mustard', 'Air Freshener']",0.477889
1,['Paper Towels'],0.138435,"['Orange', 'Pancake Mix']",0.450973
2,['Cereal'],0.130987,"['Eggs', 'Plant Fertilizer']",0.408428
3,['Chips'],0.121737,"['Peanut Butter', 'Tomatoes']",0.400357
4,['Pasta'],0.113040,"['Tuna', 'Insect Repellent']",0.387496


------------------------------------------------------------


In [22]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.evaluation import train_test_split
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

def calculate_metrics_manual(model, train_matrix: csr_matrix, test_matrix: csr_matrix, K: int = 10, num_users: int = 1000):
    """
    Кастомная функция оценки, которая обходит баги scipy/implicit.
    Считает Precision@K и NDCG@K.
    """
    # 1. Находим только тех пользователей, у которых ЕСТЬ покупки в тестовой выборке
    test_users = np.where(test_matrix.getnnz(axis=1) > 0)[0]
    
    # Чтобы не ждать вечность, берем случайную подвыборку (например, 1000 человек)
    np.random.seed(42)
    if len(test_users) > num_users:
        test_users = np.random.choice(test_users, num_users, replace=False)
        
    precisions = []
    ndcgs =[]
    
    # Проходимся по каждому пользователю
    for u in test_users:
        # Получаем Топ-K рекомендаций для пользователя
        ids, _ = model.recommend(
            userid=[u], 
            user_items=train_matrix[u], 
            N=K, 
            filter_already_liked_items=True
        )
        rec_items = ids[0] # Извлекаем список товаров из батча
        
        # Получаем реальные товары (Ground Truth), которые он купил в тесте
        true_items = set(test_matrix.indices[test_matrix.indptr[u]:test_matrix.indptr[u+1]])
        
        if len(true_items) == 0:
            continue
            
        # --- Считаем Precision@K ---
        hits = len(set(rec_items).intersection(true_items))
        precisions.append(hits / K)
        
        # --- Считаем NDCG@K ---
        dcg = 0.0
        idcg = 0.0
        
        # Считаем реальный DCG (штрафуем за позицию)
        for i, item in enumerate(rec_items):
            if item in true_items:
                dcg += 1.0 / np.log2(i + 2) # i начинается с 0, формула требует (позиция + 1)
                
        # Считаем идеальный DCG (если бы все угаданные стояли на первых местах)
        for i in range(min(len(true_items), K)):
            idcg += 1.0 / np.log2(i + 2)
            
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
        
    return np.mean(precisions), np.mean(ndcgs)

# --- Запуск ---

print("1. Разбиение данных на Train и Test...")
train_matrix, test_matrix = train_test_split(interaction_matrix, train_percentage=0.8, random_state=42)

print("2. Переобучение моделей...")
als_model = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=15, random_state=42)
als_model.fit(train_matrix)

bpr_model = BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=15, random_state=42)
bpr_model.fit(train_matrix)

print("\n3. Оценка моделей (кастомный алгоритм для 1000 случайных пользователей)...")

als_prec, als_ndcg = calculate_metrics_manual(als_model, train_matrix, test_matrix, K=10)
bpr_prec, bpr_ndcg = calculate_metrics_manual(bpr_model, train_matrix, test_matrix, K=10)

results_df = pd.DataFrame([
    {"Model": "ALS", "Test Precision@10": round(als_prec, 4), "Test NDCG@10": round(als_ndcg, 4)},
    {"Model": "BPR", "Test Precision@10": round(bpr_prec, 4), "Test NDCG@10": round(bpr_ndcg, 4)}
])

print("\n=== Результаты Честной Оценки ===")
print(results_df.to_markdown(index=False))

1. Разбиение данных на Train и Test...
2. Переобучение моделей...


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]


3. Оценка моделей (кастомный алгоритм для 1000 случайных пользователей)...

=== Результаты Честной Оценки ===
| Model   |   Test Precision@10 |   Test NDCG@10 |
|:--------|--------------------:|---------------:|
| ALS     |              0.0035 |         0.0099 |
| BPR     |              0.0008 |         0.0015 |


# Часть 2

In [1]:
import lightfm
import numpy as np
from lightfm import LightFM
from lightfm.datasets import fetch_movielens

print(f"Версия NumPy: {np.__version__}") # Должна быть 1.26.x, НЕ 2.x
print(f"Версия LightFM: {lightfm.__version__}")

# Загружаем данные Movielens
data = fetch_movielens(min_rating=5.0)

# Проверяем типы данных (LightFM любит np.float32)
# На всякий случай кастуем матрицу, чтобы избежать багов на Windows
train_matrix = data['train'].astype(np.float32)

print("Данные готовы. Обучаем модель...")

# Создаем модель с функцией потерь WARP (Weighted Approximate-Rank Pairwise)
# Это аналог BPR, но он оптимзирует верхушку рекомендаций
model = LightFM(loss='warp')

# Обучаем (СТРОГО num_threads=1 для Windows без OpenMP)
model.fit(train_matrix, epochs=30, num_threads=1)

print("✅ Обучение успешно завершено! Ядро выжило!")

c:\Users\Rog G16\.conda\envs\FU3.11\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


: 

In [2]:
data = fetch_movielens(min_rating=5.0)

In [3]:
model = LightFM(loss='warp')
model.fit(data['train'], epochs=30, num_threads=1)

: 